In [5]:
# IndiaKart: Colab data setup, quality checks, EDA charts, and KPI scorecard

import os
import re
import sys
import zipfile
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
from urllib.parse import quote

# ----------------------------
# 1. Download and extract data
# ----------------------------

REPO_OWNER = "sugumar338-ux"
REPO_NAME = "Internmo-IndiaKart-ECommerce-Analytics-Project"
BRANCH = "main"
ZIP_NAME = "44120013dcc669c383d89c8c4a40b746 (1).zip"

WORK_DIR = Path("/content/indiakart_project")
WORK_DIR.mkdir(parents=True, exist_ok=True)

required_tables = {
    "customers": "customers",
    "orders": "orders",
    "order_items": "orderitems",
    "payments": "payments",
    "returns": "returns",
    "products": "products",
    "inventory": "inventory",
    "suppliers": "suppliers",
}

def normalize_name(value):
    return re.sub(r"[^a-z0-9]", "", str(value).lower())

def find_csv_tables(root):
    """Find the expected CSV tables, including inside nested ZIP folders."""
    csv_files = list(Path(root).rglob("*.csv"))
    result = {}

    for table, wanted in required_tables.items():
        matches = []
        for path in csv_files:
            stem = normalize_name(path.stem)
            # Handles names such as "customers DA.csv" and "order_items.csv".
            if stem == wanted or stem == wanted + "da":
                matches.append(path)

        if matches:
            result[table] = matches[0]

    return result

tables_found = find_csv_tables(WORK_DIR)

if len(tables_found) < len(required_tables):
    print("CSV files were not already available. Downloading the project ZIP...")

    api_url = (
        f"https://api.github.com/repos/{REPO_OWNER}/{REPO_NAME}/contents/"
        f"{quote(ZIP_NAME)}?ref={BRANCH}"
    )
    response = requests.get(api_url, timeout=60)
    response.raise_for_status()
    file_info = response.json()

    download_url = file_info.get("download_url")
    if not download_url:
        raise RuntimeError("GitHub did not return a download URL for the dataset ZIP.")

    zip_path = Path("/content") / ZIP_NAME
    zip_response = requests.get(download_url, timeout=180)
    zip_response.raise_for_status()
    zip_path.write_bytes(zip_response.content)

    # Extract files safely, rejecting paths that would escape the destination.
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            # Resolve destination path and prevent directory traversal vulnerability
            target_path = WORK_DIR / member.filename
            try:
                target_path.resolve().relative_to(WORK_DIR.resolve())
            except ValueError:
                # Skip directory traversal attempts
                continue
            archive.extract(member, WORK_DIR)

    print("Extraction complete.")
    tables_found = find_csv_tables(WORK_DIR)
